# HaLRTC 图像修复算法
高精度低秩张量补全 (High-accuracy Low-Rank Tensor Completion)

In [ ]:
import numpy as np
from numpy.linalg import inv
from skimage.metrics import structural_similarity as ssim
from PIL import Image
import matplotlib.pyplot as plt
import os
import time

def ten2mat(tensor, mode):
    """张量按模展开为矩阵"""
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order='F')

def mat2ten(mat, tensor_size, mode):
    """矩阵折叠为张量"""
    index = list()
    index.append(mode)
    for i in range(tensor_size.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(tensor_size[index]), order='F'), 0, mode)

def svt(mat, lambda0):
    """奇异值阈值化 (SVT)"""
    u, s, v = np.linalg.svd(mat, full_matrices=0)
    vec = s - lambda0
    vec[np.where(vec < 0)] = 0
    return np.matmul(np.matmul(u, np.diag(vec)), v)

In [ ]:
def HaLRTC(I, Omega, alpha, rho, maxiter, epsilon=1e-4, verbose=True):
    """
    HaLRTC 算法主函数
    
    参数:
    I: 原始图像张量
    Omega: 二值掩码 (1表示观测, 0表示缺失)
    alpha: 核范数正则化权重
    rho: ADMM惩罚参数
    maxiter: 最大迭代次数
    epsilon: 收敛阈值
    verbose: 是否打印进度
    
    返回:
    tensor_recovery: 修复后的张量
    psnr_list: 每次迭代的PSNR值
    ssim_value: 最终SSIM值
    elapsed_time: 运行时间
    """
    Omega = Omega.astype(bool)
    dim0 = I.ndim
    dim1, dim2, dim3 = I.shape
    maxP = float(np.max(I))
    
    position = np.where(Omega == 1)
    sparse_tensor = I * Omega
    tensor_hat = sparse_tensor.copy()
    
    Z = np.zeros((dim1, dim2, dim3, dim0))
    T = np.zeros((dim1, dim2, dim3, dim0))
    
    last_tensor = tensor_hat.copy()
    train_norm = np.linalg.norm(sparse_tensor)
    psnr_list = np.zeros(maxiter)
    
    start_time = time.time()
    
    for iter in range(maxiter):
        for k in range(dim0):
            Z[:, :, :, k] = mat2ten(
                svt(ten2mat(tensor_hat + T[:, :, :, k] / rho, k), alpha / rho),
                np.array([dim1, dim2, dim3]), k
            )
        
        tensor_hat = np.mean(Z - T / rho, axis=3)
        tensor_hat[position] = sparse_tensor[position]
        
        for k in range(dim0):
            T[:, :, :, k] = T[:, :, :, k] + rho * (tensor_hat - Z[:, :, :, k])
        
        tensor_recovery = np.maximum(0, tensor_hat)
        tensor_recovery = np.minimum(maxP, tensor_recovery)
        
        mseC1 = np.linalg.norm(I[:, :, 0].astype(float) - tensor_recovery[:, :, 0], 'fro') ** 2 / (dim1 * dim2)
        psnrC1 = 10 * np.log10(maxP**2 / mseC1)
        mseC2 = np.linalg.norm(I[:, :, 1].astype(float) - tensor_recovery[:, :, 1], 'fro') ** 2 / (dim1 * dim2)
        psnrC2 = 10 * np.log10(maxP**2 / mseC2)
        mseC3 = np.linalg.norm(I[:, :, 2].astype(float) - tensor_recovery[:, :, 2], 'fro') ** 2 / (dim1 * dim2)
        psnrC3 = 10 * np.log10(maxP**2 / mseC3)
        psnr_list[iter] = (psnrC1 + psnrC2 + psnrC3) / 3

        if verbose:
            print(f"Epoch = {iter+1}, PSNR = {psnr_list[iter]:.4f} dB")
        
        tol = np.linalg.norm(tensor_hat - last_tensor) / train_norm
        last_tensor = tensor_hat.copy()
        
        if tol < epsilon:
            if verbose:
                print(f"\n在第 {iter+1} 次迭代时收敛")
            psnr_list = psnr_list[:iter+1]
            break
    
    ssim_value = ssim(I.astype(float), tensor_recovery, data_range=maxP, channel_axis=2)
    elapsed_time = time.time() - start_time
    
    if verbose:
        print(f"\n{'='*50}")
        print(f"最终PSNR: {psnr_list[-1]:.4f} dB")
        print(f"最终SSIM: {ssim_value:.4f}")
        print(f"运行时间: {elapsed_time:.2f} 秒")
        print(f"{'='*50}")
    
    return tensor_recovery, psnr_list, ssim_value, elapsed_time


In [ ]:
# 参数设置
seedr = 920
np.random.seed(seedr)

# 图像列表
image_names = ['Airplane', 'House256', 'House512', 'Peppers', 'Tree', 'Sailboat', 'Female']

# 算法参数
missing_rates = [0.8, 0.9, 0.95]
alpha = 1.0
rho = 0.01
maxiter = 1000
epsilon = 1e-4

# 存储结果
all_results = {}

for missing_rate in missing_rates:
    result_dir = f'results/HaLRTC/HaLRTC{missing_rate*100}%'
    os.makedirs(result_dir, exist_ok=True)
    results = {}

    print(f"\n{'#'*70}")
    print(f"开始处理缺失率: {missing_rate*100:.1f}%")
    print(f"{'#'*70}\n")

    for img_name in image_names:
        print(f"\n{'='*60}")
        print(f"处理图像: {img_name}")
        print(f"{'='*60}\n")
        
        try:
            I = np.array(Image.open(f'./data/{img_name}.tiff'))
            h, w, c = I.shape
            
            Omega = np.random.rand(h, w, c) > missing_rate
            
            print(f"图像尺寸: {I.shape}")
            print(f"缺失率: {missing_rate*100:.1f}%")
            print(f"观测像素: {np.sum(Omega)} / {I.size}\n")
            
            X_halrtc, psnr_halrtc, ssim_halrtc, elapsed_time = HaLRTC(I, Omega, alpha, rho, maxiter, epsilon)
            
            results[img_name] = {
                'original': I,
                'observed': I * Omega,
                'recovered': X_halrtc,
                'psnr_list': psnr_halrtc,
                'final_psnr': psnr_halrtc[-1],
                'ssim': ssim_halrtc,
                'time': elapsed_time
            }
            
            Image.fromarray(np.uint8(X_halrtc)).save(f'{result_dir}/{img_name}_修复后.png')
            
        except Exception as e:
            print(f"处理 {img_name} 时出错: {str(e)}")
            continue

    plt.rcParams['font.sans-serif'] = ['SimHei']
    for img_name in results.keys():
        data = results[img_name]
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        axes[0].imshow(np.uint8(data['original']))
        axes[0].set_title('原始图像', fontsize=14)
        axes[0].axis('off')
        
        axes[1].imshow(np.uint8(data['observed']))
        axes[1].set_title(f'观测数据 ({missing_rate*100:.0f}% 缺失)', fontsize=14)
        axes[1].axis('off')
        
        axes[2].imshow(np.uint8(data['recovered']))
        axes[2].set_title(
            f'HaLRTC 修复\nPSNR={data["final_psnr"]:.2f}dB, SSIM={data["ssim"]:.4f}\n时间={data["time"]:.2f}s',
            fontsize=14
        )
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.savefig(f'{result_dir}/{img_name}_对比.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        plt.figure(figsize=(10, 6))
        plt.plot(range(1, len(data['psnr_list'])+1), data['psnr_list'], 'b-', linewidth=2)
        plt.xlabel('迭代次数', fontsize=12)
        plt.ylabel('PSNR (dB)', fontsize=12)
        plt.title(f'{img_name} - HaLRTC 收敛曲线', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{result_dir}/{img_name}_收敛曲线.png', dpi=150, bbox_inches='tight')
        plt.show()

    print(f"\n缺失率 {missing_rate*100:.1f}% 的图像结果已保存")
    all_results[missing_rate] = results

print(f"\n{'='*60}")
print('所有缺失率处理完成！')
print(f"{'='*60}")


In [ ]:
# 单独导出 Excel 汇总
import pandas as pd

if 'all_results' not in globals():
    raise NameError('当前内核中没有 all_results，请先运行上一单元生成实验结果。')

for missing_rate in missing_rates:
    result_dir = f'results/HaLRTC/HaLRTC{missing_rate*100}%'
    results = all_results[missing_rate]

    summary_data = []
    for img_name, data in results.items():
        summary_data.append({
            '图像名称': img_name,
            'PSNR (dB)': f"{data['final_psnr']:.4f}",
            'SSIM': f"{data['ssim']:.4f}",
            '运行时间 (s)': f"{data['time']:.2f}"
        })

    df = pd.DataFrame(summary_data)
    df.to_excel(f'{result_dir}/实验总结.xlsx', index=False, engine='openpyxl')
    print(f'缺失率 {missing_rate*100:.1f}% 的 Excel 已导出到: {result_dir}/实验总结.xlsx')
